# Demographic Robustness — **Our model (cb 128/256/512/1024/2048) + 4 baselines**

`dx_age_robustness.ipynb` 와 동일한 ICD/super-class 비교를, 우리 사전학습 모델 5개 (cb size별, **v4**)에 대해 수행한다.

**v6 cb1024** (MoRyECG + GlobalRefineBlock) 도 동일 분석에 포함 — `OURS_V6_MODELS` / `OURS_ALL` 사용.

**선행 작업** (한 번만):
```bash
conda activate hbkim
cd /home/irteam/local-node-d/hbkimi/ecg-fm
# v4 (5 cb sizes)
python scripts/extract_embeddings_for_visualize.py --cb all
# v6 cb1024 (MoRyECG-v6, 별도 prefix)
python scripts/extract_embeddings_for_visualize.py \
    --cb 1024 \
    --cfg-name moryecg_heedb_cb1024_v6.yaml \
    --ckpt-suffix v6 \
    --model-prefix Ours-v6
```
→ `Ours-cb{K}_*` (v4) / `Ours-v6-cb{K}_*` (v6) 가 visualize 프로젝트의 embeddings 디렉토리에 생긴다.

> Plot 의 outlier 보정: `umap_view.quick_dx` 가 **`coord_clip_pct=99.5`** 디폴트로 동작 — UMAP 좌표 양 끝 0.25% 를 axes 밖으로 잘라 dense main blob 이 잘 보임. `coord_clip_pct=None` 으로 끄거나 `coord_clip_pct=99` 등으로 더 빡빡하게 조절 가능.

In [ ]:
import sys, os, glob, json, re
from pathlib import Path

# umap_view 는 visualize 프로젝트에 있다 (ecg-fm 내부에는 없음).
VISUALIZE_ROOT = '/home/irteam/local-node-d/tykim/visuallize'
if VISUALIZE_ROOT not in sys.path:
    sys.path.insert(0, VISUALIZE_ROOT)

from scripts.umap_view import (
    quick_dx, quick_dx_raw, get_dx_codes, get_super_codes,
    ICD_DISPLAY, SUPER_DISPLAY, ICD_LABEL_MAP, SUPER_LABEL_MAP,
    list_models, dx_compatibility_table,
    DEFAULT_EMB_DIR,
)
import numpy as np
import pandas as pd
pd.set_option('display.width', 140)
pd.set_option('display.max_rows', 60)

EMB_DIR = Path(DEFAULT_EMB_DIR)
print('emb_dir:', EMB_DIR)
print('available models (default discover):', list_models())

# 우리 모델 리스트 — v4 (Ours-cb{K}) + v6 (Ours-v6-cb{K}) 모두 글롭.
# meta.json 파일명 패턴: Ours-cb{K}_meta.json  (v4)
#                       Ours-v6-cb{K}_meta.json (v6, 신규)
def _scan_ours_models(glob_pat: str, name_filter):
    out = []
    for p in glob.glob(str(EMB_DIR / glob_pat)):
        name = Path(p).stem.replace('_meta', '')
        if name_filter(name):
            out.append(name)
    # cb 숫자순 정렬
    return sorted(out, key=lambda n: int(re.search(r'cb(\d+)', n).group(1)))

# v4: 'Ours-cb{K}' (prefix 에 'v6' 없음)
OURS_MODELS    = _scan_ours_models('Ours-cb*_meta.json',
                                   lambda n: '-v6-' not in n)
# v6: 'Ours-v6-cb{K}'
OURS_V6_MODELS = _scan_ours_models('Ours-v6-cb*_meta.json', lambda n: True)
# v4 + v6 합본 (한 figure 에서 나란히 보고 싶을 때)
OURS_ALL = OURS_MODELS + OURS_V6_MODELS

print('our v4 models:', OURS_MODELS)
print('our v6 models:', OURS_V6_MODELS)

## 1. ICD 호환성 표 — `dx_age_robustness.ipynb` 와 동일

PTBXL/ZZU 양쪽에 sample이 ≥1 개씩 존재하는 진단 코드를 cross-dataset 비교 후보로 사용.

In [ ]:
compat = dx_compatibility_table()
compat_codes = compat[compat['compatible']]['icd'].tolist()
print('compatible ICD codes:', compat_codes)
compat[compat['compatible']]

## 2. **우리 모델만** — cb 128 → 2048 비교 (호환 12개 진단)

행 = cb size, 열 = (combined / adult ≥18 / pediatric <18 / metrics).
각 cell 에서 해당 그룹만 진단 색칠, 나머지는 회색.

In [ ]:
fig_ours, metrics_ours = quick_dx(
    models=OURS_ALL,
    include_codes=compat_codes,
    age_split=18.0,
    balance_per_code=500,
)

In [ ]:
# 핵심 부정맥/전도장애 focused
fig_ours_arr, metrics_ours_arr = quick_dx(
    models=OURS_ALL,
    include_codes=['I45.1', 'I44.0', 'I44.4', 'I45.6', 'I45.81', 'I47.1'],
    age_split=18.0,
    balance_per_code=300,
)

In [ ]:
# Normal 제외 — 비정상 진단만 색칠 (Normal 회색)
fig_ours_nn, metrics_ours_nn = quick_dx(
    models=OURS_ALL,
    include_codes=compat_codes,
    exclude_codes=['Z00.00'],
    age_split=18.0,
)

## 3. **우리 모델만** — PTBXL super class (NORM/MI/STTC/CD/HYP)

양쪽 그룹의 sample 수를 자동 균등화 (`balance_strict_groups=True`).

In [ ]:
fig_ours_super, metrics_ours_super = quick_dx(
    models=OURS_ALL,
    code_scheme='super',
    include_codes=['NORM', 'STTC', 'CD', 'HYP', 'MI'],
    age_split=18.0,
    balance_strict_groups=True,
)

## 4. Subgroup gap 표 — adult vs pediatric BACC

`gap` ↓ + `worst` ↑ 가 좋다. cb size 별로 어떤 게 가장 robust한지 본다.

In [ ]:
def subgroup_table(metrics_dict, display_map=None):
    """quick_dx metrics → adult/ped BACC, gap, worst 표."""
    if display_map is None:
        display_map = ICD_DISPLAY
    by_grp_first = next(iter(metrics_dict.values()))
    adult_key = next(k for k in by_grp_first if 'adult' in k)
    ped_key   = next(k for k in by_grp_first if 'pediatric' in k)
    rows = []
    for model, by_grp in metrics_dict.items():
        for code, mm_a in by_grp.get(adult_key, {}).items():
            mm_p = by_grp.get(ped_key, {}).get(code, {})
            ba, bp = mm_a.get('bacc'), mm_p.get('bacc')
            if ba is None or bp is None or not (np.isfinite(ba) and np.isfinite(bp)):
                continue
            rows.append({
                'model': model, 'code': code,
                'dx': display_map.get(code, code),
                'adult_bacc': round(ba, 3),
                'pedi_bacc':  round(bp, 3),
                'gap':        round(abs(ba - bp), 3),
                'worst':      round(min(ba, bp), 3),
                'n_adult':    int(mm_a.get('n_pos', 0)),
                'n_pedi':     int(mm_p.get('n_pos', 0)),
            })
    return pd.DataFrame(rows)

df_ours = subgroup_table(metrics_ours)
summary_ours = df_ours.groupby('model')[['gap','worst']].mean().round(3)
summary_ours = summary_ours.reindex(OURS_ALL)
summary_ours['rank_gap']   = summary_ours['gap'].rank()
summary_ours['rank_worst'] = summary_ours['worst'].rank(ascending=False)
print('=== Ours: cb size별 평균 (gap↓ / worst↑) — 호환 12개 진단 ===')
print(summary_ours)

In [ ]:
# 진단별 상세
print('=== gap (낮을수록 좋음) ===')
print(df_ours.pivot_table(index='dx', columns='model', values='gap')[OURS_ALL].round(3))
print('\n=== worst-group BACC (높을수록 좋음) ===')
print(df_ours.pivot_table(index='dx', columns='model', values='worst')[OURS_ALL].round(3))

In [ ]:
# super class subgroup 표
df_ours_super = subgroup_table(metrics_ours_super, display_map=SUPER_DISPLAY)
print('=== Ours super: 모델·super 진단별 ===')
print(df_ours_super[['model','dx','adult_bacc','pedi_bacc','gap','worst']].to_string(index=False))
print('\n=== cb size별 평균 ===')
print(df_ours_super.groupby('model')[['gap','worst']].mean().reindex(OURS_ALL).round(3))

## 5. **우리 모델 + 4 baselines** — 전체 비교

행 수가 많아서 figure 크기 ↑. 빠르게 보고 싶으면 위의 ours-only 셀만 사용.

In [ ]:
BASELINES = list_models()  # ['CPC','ECG-FM','ECG-Founder','ECG-JEPA']
ALL_MODELS = BASELINES + OURS_ALL
print('all models:', ALL_MODELS)

fig_all, metrics_all = quick_dx(
    models=ALL_MODELS,
    include_codes=compat_codes,
    age_split=18.0,
    balance_per_code=500,
)

In [ ]:
df_all = subgroup_table(metrics_all)
summary_all = df_all.groupby('model')[['gap','worst']].mean().round(3)
summary_all = summary_all.reindex(ALL_MODELS)
summary_all['rank_gap']   = summary_all['gap'].rank()
summary_all['rank_worst'] = summary_all['worst'].rank(ascending=False)
print('=== 전체 모델 평균 (호환 12개 진단) ===')
print(summary_all)

In [ ]:
# super class — 전체 모델
fig_all_super, metrics_all_super = quick_dx(
    models=ALL_MODELS,
    code_scheme='super',
    include_codes=['NORM','STTC','CD','HYP','MI'],
    age_split=18.0,
    balance_strict_groups=True,
)

In [ ]:
df_all_super = subgroup_table(metrics_all_super, display_map=SUPER_DISPLAY)
summary_all_super = df_all_super.groupby('model')[['gap','worst']].mean().round(3)
summary_all_super = summary_all_super.reindex(ALL_MODELS)
summary_all_super['rank_gap']   = summary_all_super['gap'].rank()
summary_all_super['rank_worst'] = summary_all_super['worst'].rank(ascending=False)
print('=== 전체 모델 평균 (super 5개 진단) ===')
print(summary_all_super)

## 6. (보너스) 우리 모델의 raw paper-label UMAP — 단일 데이터셋

ICD 통합 없이 PTBXL/chapman paper 라벨 약자 그대로 색칠 (한 모델만).

In [ ]:
PICK = OURS_ALL[5] if OURS_ALL else None
if PICK:
    quick_dx_raw(
        PICK, 'ptbxl',
        label_columns=['AFIB','SR','CRBBB','CLBBB','PVC','PAC','STACH'],
        max_per_label=300,
    )

## 7. v4 cb1024 vs **v6 cb1024** head-to-head

같은 codebook size (1024) 에서 v4 (`masked_beat_*`) → v6 (`moryecg_*` + GlobalRefineBlock) 변경의 효과.

In [ ]:
V4V6 = [m for m in OURS_MODELS if m.endswith('cb1024')] + OURS_V6_MODELS
print('v4 vs v6:', V4V6)

# (a) ICD compatible 12개 진단 — combined / adult / pediatric
fig_v4v6, metrics_v4v6 = quick_dx(
    models=V4V6,
    include_codes=compat_codes,
    age_split=18.0,
    balance_per_code=500,
)

# (b) PTBXL super class
fig_v4v6_super, metrics_v4v6_super = quick_dx(
    models=V4V6,
    code_scheme='super',
    include_codes=['NORM', 'STTC', 'CD', 'HYP', 'MI'],
    age_split=18.0,
    balance_strict_groups=True,
)

# (c) subgroup gap 표
df_v4v6 = subgroup_table(metrics_v4v6)
print('=== v4 cb1024 vs v6 cb1024 — 진단 평균 (gap↓ / worst↑) ===')
print(df_v4v6.groupby('model')[['gap','worst']].mean().reindex(V4V6).round(3))